# **Сбор всех необходимых метрик и данных для дашборда с помощью CatBoost модели**

**Проект:** Анализ и визуализация данных с использованием Yandex DataLens: исследование по прогнозированию CTR

**Автор:** Грицан М.А., студент группы БПМИ-247, 2 курса

**Дата:** 26-03-2026  

**Цель:** Обучение CatBoost модели, примение модели на тестовой выборке для получения результатов в kaggle соревновании ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/overview), а также сбор всех необходимых для дашборда метрик

#### Необходимые библиотеки и настройка графиков:

In [1]:
%pip install catboost -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import zipfile

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.calibration import calibration_curve

In [3]:
%config InlineBackend.figure_format = 'retina'

sns.set(style='darkgrid', palette='deep')

plt.rcParams['figure.figsize'] = 8, 5
plt.rcParams['font.size'] = 12
plt.rcParams['savefig.format'] = 'pdf'

### 0. Загрука набора данных c kaggle
Скачиваем все файлы с соревнования ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/data) для дальнейшего использования.

In [ ]:
os.environ['KAGGLE_API_TOKEN'] = "ВАШ_KAGGLE_API_TOKEN"

print("✅ Kaggle API ключ установлен!")

✅ Kaggle API ключ установлен!


In [5]:
%pip install kaggle -q
!kaggle competitions download -c avazu-ctr-prediction -p ../

Note: you may need to restart the kernel to use updated packages.
avazu-ctr-prediction.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
with zipfile.ZipFile('../avazu-ctr-prediction.zip', 'r') as zip_ref:
    zip_ref.extractall('../avazu-ctr-prediction')

### 1. Считывание валидационной выборки и применение модели

In [ ]:
def data_tranformer(df: pd.DataFrame):
    dt = pd.to_datetime(df['hour'], format='%y%m%d%H')
    df['day_of_week'] = dt.dt.dayofweek
    df['hour_of_day'] = dt.dt.hour

    return df

target = 'click'
categorical_features = [
    'C1', 'banner_pos', 'site_id', 'site_domain', 'site_category',
    'app_id', 'app_domain', 'app_category', 'device_id', 'device_ip',
    'device_model', 'device_type', 'device_conn_type',
    'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21',
    'day_of_week', 'hour_of_day'
]

In [ ]:
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)

train_file = '../avazu-ctr-prediction/train.gz'
print(f"⏳ Загрузка валиадационной выборки из '{train_file}'...")

val_df = pd.read_csv(
    train_file,
    compression='gzip',
    skiprows=range(1, 30000001),
)
print(f"Прочитано {len(val_df)} строк, {len(val_df.columns)} колонки")

⏳ Загрузка валиадационной выборки из '../avazu-ctr-prediction/train.gz'...


KeyboardInterrupt: 

In [ ]:
os.makedirs(f'models', exist_ok=True)
model_path = 'models/catboost_ctr_model.cbm'

print("⏳ Достаем обученную модель...")
model = CatBoostClassifier()
model.load_model(model_path)

print(f"Количество признаков в модели: {len(model.feature_names_)}")

⏳ Достаем обученную модель...
Количество признаков в модели: 23


А теперь применим модель к валидационной выборке:

In [ ]:
val_df = data_tranformer(val_df)
val_X = val_df[categorical_features]

predicted_target = "predicted_click"
predicted_ctr = "predicted_ctr"

val_df[predicted_ctr] = model.predict_proba(val_X)[:, 1]
val_df[predicted_target] = model.predict(val_X)

### 2. Сбор всех необходимых метрик и данных для дашборда

Решение отправленно, теперь надо подготовить и сохранить все данные для посторения 3-й вкладки дашборда в Yandex DataLens.

##### Базовые метрики на валидационной выборке:

In [ ]:
print("⏳ Расчет ROC-AUC и LogLoss на валидационной выборке...")

roc_auc = roc_auc_score(val_df[target], val_df[predicted_ctr])
logloss = log_loss(val_df[target], val_df[predicted_ctr])

metrics_df = pd.DataFrame({
    'metric': ['ROC-AUC', 'Log Loss'],
    'value': [roc_auc, logloss]
})
metrics_df.to_csv('data/model_metrics.csv', index=False)
print("✅ Базовые метрики сохранены: data/model_metrics.csv")

metrics_df

⏳ Расчет ROC-AUC и LogLoss на валидационной выборке...
✅ Базовые метрики сохранены: data/model_metrics.csv


,metric,value
0,ROC-AUC,0.707386
1,Log Loss,0.419578


##### Данные для Lift:

In [ ]:
print("⏳ Расcчет данных для Lift-таблицы...")

val_pool = Pool(val_df[categorical_features], cat_features=categorical_features)
all_shap_values = model.get_feature_importance(val_pool, type='ShapValues')

shap_values_only = all_shap_values[:, :-1]
shap_df = pd.DataFrame(shap_values_only, columns=categorical_features)[categorical_features]


total_traffic = len(val_df)
shap_lift_data = []

for category_name in categorical_features:
    temp_df = pd.DataFrame({
        'category_value': val_df[category_name].values,
        'shap_value': shap_df[category_name].values
    })

    grouped = temp_df.groupby('category_value').agg(
        traffic=('shap_value', 'count'),
        avg_shap=('shap_value', 'mean')
    ).reset_index()

    grouped['category_name'] = category_name
    grouped['traffic_share'] = (grouped['traffic'] / total_traffic)

    shap_lift_data.append(grouped[['category_name', 'category_value', 'avg_shap', 'traffic_share']])

shap_lift_df = pd.concat(shap_lift_data, ignore_index=True)

shap_lift_df.to_csv('data/shap_lift.csv', index=False)
print("✅ Lift-таблица на основе SHAP значений сохранена в data/shap_lift.csv")

shap_lift_df

⏳ Расcчет данных для Lift-таблицы...
✅ Lift-таблица на основе SHAP значений сохранена в data/shap_lift.csv


,category_name,category_value,avg_shap,traffic_share
0,C1,1001,-0.018522,0.000171
1,C1,1002,-0.002749,0.049125
2,C1,1005,0.000500,0.927800
3,C1,1007,-0.004526,0.000516
4,C1,1008,-0.001761,0.000039
...,...,...,...,...
2965616,hour_of_day,19,-0.074294,0.044178
2965617,hour_of_day,20,-0.102385,0.038150
2965618,hour_of_day,21,-0.094152,0.035881
2965619,hour_of_day,22,-0.094208,0.030282


##### Feature Importance:

In [ ]:
print("⏳ Получение feature_importance обученной модели...")

feature_importance = model.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': model.feature_names_,
    'importance': feature_importance
})

importance_df.to_csv('data/feature_importance.csv', index=False)
print("✅ Важность признаков сохранена в data/feature_importance.csv")

importance_df.sample(5)

⏳ Получение feature_importance обученной модели...
✅ Важность признаков сохранена в data/feature_importance.csv


,feature,importance
13,C14,6.873253
16,C17,2.563066
8,device_id,10.065254
22,hour_of_day,0.785875
12,device_conn_type,0.415396


##### Топ связок признаков, которые выделил CatBoost:

In [ ]:
print("⏳ Получение качества связок признаков обученной модели...")

interaction_importance = model.get_feature_importance(type="Interaction")

interactions_df = pd.DataFrame(
    interaction_importance,
    columns=['feature_1_idx', 'feature_2_idx', 'interaction_score']
)

feature_names = model.feature_names_
interactions_df['feature_1'] = interactions_df['feature_1_idx'].apply(lambda x: feature_names[int(x)])
interactions_df['feature_2'] = interactions_df['feature_2_idx'].apply(lambda x: feature_names[int(x)])
interactions_df = interactions_df[['feature_1', 'feature_2', 'interaction_score']]

interactions_df.to_csv('data/feature_interactions.csv', index=False)
print("✅ Cвязки признаков с качеством успешно сохранены в data/feature_interactions.csv")

interactions_df.sample(5)

⏳ Получение качества связок признаков обученной модели...
✅ Cвязки признаков с качеством успешно сохранены в data/feature_interactions.csv


,feature_1,feature_2,interaction_score
165,C1,C14,0.073357
212,C1,C19,0.010602
169,device_model,C15,0.067300
81,C14,hour_of_day,0.324876
26,device_ip,C19,1.035747


##### Предсказанный CTR в сравнении с реальным в срезах разных фичей:

In [ ]:
diff_ctr_data = []

needable_categorical_features = ['site_category', 'app_category', 'device_model', 'device_type']

for category_name in needable_categorical_features:
    grouped = val_df.groupby(category_name).agg({target: 'mean', predicted_ctr: 'mean'}).reset_index()
    grouped.columns = ['category_value', 'real_ctr', 'predicted_ctr']
    grouped['category_name'] = category_name

    diff_ctr_data.append(grouped)

diff_ctr_df = pd.concat(diff_ctr_data, ignore_index=True)

diff_ctr_df.to_csv('data/diff_ctr.csv', index=False)
print("✅ Данные для графиков перепредсказонного и недопредсказанного CTR успешно сохранены в data/diff_ctr.csv")

diff_ctr_df.sample(5)

✅ Данные для графиков перепредсказонного и недопредсказанного CTR успешно сохранены в data/diff_ctr.csv


,category_value,real_ctr,predicted_ctr,category_name
2702,6a12146c,0.296178,0.201274,device_model
3247,7f4690d6,0.164894,0.202644,device_model
4596,b552ce88,0.000000,0.072732,device_model
3257,7f79bd09,0.181818,0.152662,device_model
6005,eb7fc71c,0.153846,0.139392,device_model


##### Данные для диаграммы надежности модели:

In [ ]:
print("⏳ Расчет данных для диаграммы надежности (Calibration Curve)...")

prob_true, prob_pred = calibration_curve(val_df[target], val_df[predicted_ctr], n_bins=10, strategy='uniform')

calibration_df = pd.DataFrame({
    'predicted_ctr': prob_pred,
    'real_ctr': prob_true
})
calibration_df['perfect_calibration'] = calibration_df['real_ctr']

calibration_df.to_csv('data/calibration_curve.csv', index=False)
print(f"\n✅ Данные для диаграммы надежности успешно сохранены в data/calibration_curve.csv")

calibration_df.sample(5)

⏳ Расчет данных для диаграммы надежности (Calibration Curve)...

✅ Данные для диаграммы надежности успешно сохранены в data/calibration_curve.csv


,predicted_ctr,real_ctr,perfect_calibration
5,0.544925,0.429205,0.429205
4,0.441728,0.357539,0.357539
2,0.248585,0.224600,0.224600
8,0.850074,0.447714,0.447714
6,0.647509,0.354233,0.354233
